In [19]:
import os
from pathlib import Path
import torch 
import numpy as np

from ml.predict import load_checkpoint

## On charge le modèle MAEv1 après premier run de préentrainement
base = Path.cwd()

ckpt_dir = Path(os.path.join(base, '..', '..', 'outputs', 'ml', 'pretrain', '2026-08-04_14-58-23', 'checkpoints', 'best.pt'))

device = torch.device("cpu")
pretrained_model, cfg, mean, scale = load_checkpoint(ckpt_dir, device)
print(pretrained_model)


TimeSeriesMAE(
  (patch_embedding): PatchEmbedding(
    (proj): Conv1d(11, 64, kernel_size=(12,), stride=(12,))
  )
  (encoder_pos_embedding): LearnedPositionalEmbedding(
    (PE): Embedding(8, 64)
  )
  (decoder_pos_embedding): LearnedPositionalEmbedding(
    (PE): Embedding(9, 32)
  )
  (encoder_blocks): ModuleList(
    (0-3): 4 x TransformerBlock(
      (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True, bias=True)
      (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True, bias=True)
      (MSA): MultiHeadSelfAttention(
        (query_matrix): Linear(in_features=64, out_features=64, bias=True)
        (key_matrix): Linear(in_features=64, out_features=64, bias=True)
        (value_matrix): Linear(in_features=64, out_features=64, bias=True)
        (output_proj): Linear(in_features=64, out_features=64, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (MLP): MLP(
        (dense1): Linear(in_features=64, out_features=256, bias=True)
        (d

In [20]:
## on charge les poids de l'encoder du modèle préentrainé sur un backbone avec les mêmes paramètres (cf MAEv1.yaml pour l'archi de l'encoder)
from ml.model import VanillaViT

encoder_sd = pretrained_model.encoder_state_dict() ## donne directement les poids de l'encodeur avec les bons mapping


backbone = VanillaViT(
    n_features = 11, 
    n_epochs= 96, 
    patch_size=12, 
    embed_dim=64,
    n_attn_heads=4,
    n_blocks=4,
    expansion_factor=4,
    dropout_rate=0.1
)

missing, unexpected = backbone.load_state_dict(encoder_sd, strict=False) # les poids sont bien chargés. 



In [27]:
from ml.datahandler import load_spacetrack_objects, build_features, diff_cols_spacetrack
data_dir = Path(os.path.join(base, '..' , '..', 'data', 'raw', 'spacetrack'))
objects = load_spacetrack_objects(data_dir)

per_obj = {}
for key, df in objects.items(): 
    features, feature_cols = build_features(df, spacetrack=True)
    X = (features[feature_cols].to_numpy(np.float32) - mean) / scale
    per_obj[key] = X

backbone.eval()
backbone.to(device)

window_size = 96 
batch_size = 256

out = {}
for norad, X in per_obj.items():
    if len(X) < window_size : 
        continue 
    windows = np.lib.stride_tricks.sliding_window_view(X, window_size, axis=0) # (L,F,W)
    representations = []

    for i in range(0, len(windows), batch_size): 
        x = torch.from_numpy(np.ascontiguousarray(windows[i:i+batch_size])).float()
        x = x.to(device)

        with torch.no_grad(): 
            representation = backbone(x)

        representations.append(representation.cpu().numpy())

    if representations:
        out[norad] = np.concatenate(representations, axis=0)
        print(out[norad].shape)


(667, 9, 64)
(104, 9, 64)
(176, 9, 64)
(771, 9, 64)
(615, 9, 64)
(788, 9, 64)
(689, 9, 64)
(8, 9, 64)
(30, 9, 64)
(612, 9, 64)
(685, 9, 64)
(584, 9, 64)
(607, 9, 64)
(655, 9, 64)
(672, 9, 64)
(615, 9, 64)
(710, 9, 64)
(451, 9, 64)
(130, 9, 64)
(103, 9, 64)
(2, 9, 64)
(469, 9, 64)
(986, 9, 64)
(756, 9, 64)
(949, 9, 64)
(525, 9, 64)
(925, 9, 64)
(757, 9, 64)
(443, 9, 64)
(93, 9, 64)
(769, 9, 64)
(121, 9, 64)
(675, 9, 64)
(689, 9, 64)
(793, 9, 64)
(211, 9, 64)
(686, 9, 64)
(661, 9, 64)
(630, 9, 64)
(665, 9, 64)
(742, 9, 64)
(946, 9, 64)
(686, 9, 64)
(674, 9, 64)
(657, 9, 64)
(6, 9, 64)
(649, 9, 64)
(663, 9, 64)
(674, 9, 64)
(677, 9, 64)
(756, 9, 64)
(130, 9, 64)
(79, 9, 64)
(713, 9, 64)
(693, 9, 64)
(75, 9, 64)
(250, 9, 64)
(22, 9, 64)
(642, 9, 64)
(662, 9, 64)
(688, 9, 64)
(793, 9, 64)
(674, 9, 64)
(643, 9, 64)
(675, 9, 64)
(671, 9, 64)
(692, 9, 64)
(662, 9, 64)
(683, 9, 64)
(637, 9, 64)
(756, 9, 64)
(488, 9, 64)
(662, 9, 64)
(689, 9, 64)
(675, 9, 64)
(923, 9, 64)
(645, 9, 64)
(34, 9, 64